# Mage: Pipeline de NYC Taxi con un Orquestador Visual

En este notebook exploramos **Mage**, un orquestador alternativo a Prefect, implementando
el **mismo pipeline de prediccion de duracion de viajes de NYC Taxi**.

El objetivo pedagogico es entender que la **orquestacion** es un concepto,
y que existen multiples herramientas para implementarlo, cada una con su filosofia.

| Aspecto | Prefect | Mage |
|---------|---------|------|
| **Filosofia** | Code-first (decoradores) | UI-first (bloques visuales) |
| **Unidad basica** | `@flow` + `@task` | Bloques: data_loader, transformer, data_exporter |
| **Edicion** | Editor de texto / IDE | UI web tipo notebook |
| **Pipeline mismo** | Exactamente el mismo | Exactamente el mismo |

## 1. Que es Mage?

**Mage** es un framework de orquestacion open-source que permite construir pipelines de datos
y ML usando una **interfaz visual tipo notebook** en el navegador.

### Conceptos clave de Mage

| Concepto | Descripcion | Equivalente en Prefect |
|----------|-------------|------------------------|
| **Block** | Unidad de trabajo (un archivo .py) | `@task` |
| **Pipeline** | Secuencia de bloques conectados | `@flow` |
| **Data Loader** | Bloque que carga datos (entrada) | Task de carga |
| **Transformer** | Bloque que transforma datos | Task de transformacion |
| **Data Exporter** | Bloque que exporta/guarda resultados | Task de salida |
| **Trigger** | Disparo programado del pipeline | `cron` / `.serve()` |

### Arquitectura del proyecto Mage

```
nyc_taxi_project/
├── metadata.yaml              # Config del proyecto
├── io_config.yaml             # Config de conexiones (DB, cloud)
├── pipelines/
│   └── nyc_taxi_training/
│       └── metadata.yaml      # Bloques y dependencias del pipeline
├── data_loaders/
│   └── load_taxi_data.py      # @data_loader
├── transformers/
│   ├── validate_data.py       # @transformer
│   └── create_features.py     # @transformer
├── data_exporters/
│   └── train_model.py         # @data_exporter
└── utils/
    └── constants.py            # Constantes compartidas
```

## 2. El Pipeline: Mismo Problema, Diferente Herramienta

Implementamos exactamente el mismo pipeline que en Prefect:

```
load_taxi_data --> validate_data --> create_features --> train_model
  (data_loader)    (transformer)     (transformer)     (data_exporter)
```

### Flujo de datos:
1. **load_taxi_data**: Descarga parquet de NYC TLC, calcula duracion, filtra
2. **validate_data**: Verifica volumen y nulos (warnings, no errores)
3. **create_features**: DictVectorizer sobre PULocationID/DOLocationID
4. **train_model**: Optuna (20 trials) + XGBoost + MLflow tracking

## 3. Codigo de cada Bloque

### 3.1 Data Loader: `load_taxi_data.py`

El decorador `@data_loader` indica que este bloque es el **punto de entrada** del pipeline.
No recibe datos de ningun bloque anterior.

In [ ]:
# Veamos el codigo del data loader
with open('nyc_taxi_project/data_loaders/load_taxi_data.py', 'r') as f:
    print(f.read())

**Diferencia clave vs Prefect:**
- Prefect: `@task(retries=3, cache_expiration=timedelta(hours=24))`
- Mage: `@data_loader` (retries y cache se configuran en la UI o metadata, no en el decorador)

En Mage, la logica de reintentos y cache se maneja a nivel de **pipeline**, no de bloque individual.

### 3.2 Transformer: `validate_data.py`

El decorador `@transformer` recibe como primer argumento el **output del bloque anterior**.

In [ ]:
with open('nyc_taxi_project/transformers/validate_data.py', 'r') as f:
    print(f.read())

### 3.3 Transformer: `create_features.py`

Crea las matrices sparse con DictVectorizer. Retorna una tupla que el siguiente bloque consume.

In [ ]:
with open('nyc_taxi_project/transformers/create_features.py', 'r') as f:
    print(f.read())

### 3.4 Data Exporter: `train_model.py`

El paso final combina Optuna + XGBoost + MLflow. El decorador `@data_exporter` indica
que este bloque produce la **salida final** del pipeline.

In [ ]:
with open('nyc_taxi_project/data_exporters/train_model.py', 'r') as f:
    print(f.read())

### 3.5 Metadata del Pipeline

El archivo `metadata.yaml` define las **conexiones entre bloques** (el grafo del pipeline).
Esto es lo que Mage usa para renderizar el pipeline visualmente en la UI.

In [ ]:
with open('nyc_taxi_project/pipelines/nyc_taxi_training/metadata.yaml', 'r') as f:
    print(f.read())

## 4. Como Ejecutar el Pipeline

### Opcion 1: UI visual (recomendado para aprender)

```bash
# Desde 03-Orchestration/Mage-pipelines/
chmod +x setup_and_run.sh
./setup_and_run.sh
```

Esto:
1. Crea un entorno virtual aislado con `uv` (no afecta tu env principal)
2. Instala mage-ai y dependencias desde `pyproject.toml`
3. Abre la UI en **http://localhost:6789**

En la UI puedes:
- Ver el pipeline como bloques conectados
- Ejecutar bloques individuales
- Ver logs en tiempo real
- Editar codigo directamente

### Opcion 2: Ejecucion por linea de comandos

```bash
./setup_and_run.sh --run
```

### Opcion 3: Ejecucion programatica

```python
import mage_ai
mage_ai.run('nyc_taxi_training', project_path='./nyc_taxi_project')
```

> **Nota sobre dependencias:** Mage requiere un entorno virtual separado porque
> sus dependencias (ej: SQLAlchemy 1.4) entran en conflicto con Prefect y MLflow
> modernos. El script usa `uv` para crear el entorno aislado, consistente con
> el tooling del resto del repositorio. Las dependencias estan declaradas en un
> `pyproject.toml` dedicado dentro de `Mage-pipelines/`.
>
> Esto es comun en la industria: diferentes herramientas en diferentes entornos.

## 5. Comparacion Lado a Lado: Prefect vs Mage

### 5.1 El mismo bloque en ambas herramientas

| Aspecto | Prefect | Mage |
|---------|---------|------|
| **Cargar datos** | `@task` con `retries=3, cache_expiration=24h` | `@data_loader` |
| **Validar** | `@task` que recibe df como argumento | `@transformer` que recibe output anterior |
| **Features** | `@task` que retorna tupla | `@transformer` que retorna tupla |
| **Entrenar** | `@task` con `retries=2` | `@data_exporter` |
| **Orquestar** | `@flow` que llama a los tasks | `metadata.yaml` que define las conexiones |
| **Deploy** | `flow.serve(cron='...')` | Triggers en UI o config YAML |

### 5.2 Codigo equivalente

```python
# ---- PREFECT ----                    # ---- MAGE ----
@task(retries=3)                       @data_loader
def load_data(year, month):            def load_data(*args, **kwargs):
    df = pd.read_parquet(url)              df = pd.read_parquet(url)
    return df                              return df

@flow                                  # (definido en metadata.yaml)
def pipeline():                        # pipeline: nyc_taxi_training
    df = load_data(2025, 1)            #   load_data -> validate -> ...
    df = validate(df)
    train(df)
```

---

## 6. Panorama de Orquestadores: Comparacion Detallada

### 6.1 Popularidad y Comunidad (datos abril 2025)

| Herramienta | GitHub Stars | Primer Release | Lenguaje | Licencia |
|-------------|-------------|----------------|----------|----------|
| **Apache Airflow** | ~38,000 | 2014 (Airbnb) | Python | Apache 2.0 |
| **Prefect** | ~18,000 | 2018 | Python | Apache 2.0 |
| **Dagster** | ~12,000 | 2019 | Python | Apache 2.0 |
| **Mage** | ~8,000 | 2022 | Python | Apache 2.0 |
| **Luigi** | ~17,500 | 2012 (Spotify) | Python | Apache 2.0 |
| **Kestra** | ~14,000 | 2022 | Java/YAML | Apache 2.0 |

> **Nota:** Las estrellas de GitHub son un indicador de popularidad, no necesariamente de calidad.
> Airflow tiene mas estrellas porque lleva mas tiempo y es el estandar de la industria.

### 6.2 Ventajas y Desventajas

#### Apache Airflow
| Ventajas | Desventajas |
|----------|-------------|
| Estandar de la industria, ampliamente adoptado | Curva de aprendizaje pronunciada |
| Ecosistema masivo (+2000 operadores/plugins) | Setup pesado (scheduler + webserver + DB) |
| Excelente para workflows complejos y ETL | No es nativo para ML (adaptado con extensiones) |
| Gran comunidad y soporte empresarial | DAGs pueden volverse dificiles de mantener |
| Integraciones con todos los cloud providers | Testing local mas complejo |

#### Prefect
| Ventajas | Desventajas |
|----------|-------------|
| Python-nativo: decoradores simples (`@flow`, `@task`) | Menos adopcion empresarial que Airflow |
| Setup minimo: `pip install prefect` y funciona | Prefect Cloud tiene limitaciones en tier gratuito |
| Retries, caching, scheduling integrados | Documentacion puede ser confusa entre v1 y v2/v3 |
| Dashboard moderno y artefactos visuales | Menos plugins que Airflow |
| Excelente para pipelines de ML | Relativamente nuevo en produccion empresarial |

#### Mage
| Ventajas | Desventajas |
|----------|-------------|
| UI visual intuitiva (ideal para principiantes) | Comunidad mas pequeña (~8k stars) |
| Edicion tipo notebook en el navegador | Menos maduro para produccion a gran escala |
| Bloques reutilizables y modulares | Conflictos de dependencias frecuentes |
| Setup rapido: `pip install mage-ai && mage start` | Documentacion menos completa |
| Tests integrados con `@test` en cada bloque | Modelo de negocio/sostenibilidad menos claro |
| Bueno para prototipos rapidos | El enfoque visual puede no escalar en equipos grandes |

#### Dagster
| Ventajas | Desventajas |
|----------|-------------|
| Modelo "assets-first" moderno y elegante | Concepto de assets puede confundir a principiantes |
| Excelente developer experience | Menos adopcion que Airflow/Prefect |
| Testing de primera clase | Mas orientado a data engineering que ML |
| Buen manejo de dependencias y tipos | Curva de aprendizaje media |

### 6.3 Cuando usar cada herramienta?

| Escenario | Herramienta Recomendada | Razon |
|-----------|------------------------|-------|
| Empresa grande con infraestructura existente | **Airflow** | Estandar, soporte empresarial, integraciones |
| Equipo de ML/Data Science en startup | **Prefect** | Setup rapido, Python-nativo, buena DX |
| Prototipo rapido o enseñanza | **Mage** | Visual, interactivo, baja curva de entrada |
| Data engineering con contratos de datos | **Dagster** | Assets-first, testing, tipado |
| Proyecto personal o aprendizaje | **Prefect o Mage** | Faciles de instalar y usar localmente |

### 6.4 Arquitectura comparada

```
PREFECT                               MAGE
======                                ====

Tu codigo Python                      Tu codigo Python
    |                                     |
    v                                     v
@flow + @task                         @data_loader / @transformer / @data_exporter
    |                                     |
    v                                     v
Prefect Server/Cloud                  Mage Server (localhost:6789)
    |                                     |
    v                                     v
Dashboard (estado, logs,              Dashboard (bloques visuales,
 artefactos, scheduling)               ejecucion, logs, scheduling)
```

**La diferencia fundamental:** En Prefect tu defines el flujo en codigo Python.
En Mage, el flujo se define visualmente (o en `metadata.yaml`) y cada bloque
es un archivo independiente.

---

## 7. Referencias y Recursos

### Documentacion oficial
- **Prefect**: https://docs.prefect.io/
- **Mage**: https://docs.mage.ai/
- **Apache Airflow**: https://airflow.apache.org/docs/
- **Dagster**: https://docs.dagster.io/

### Repositorios GitHub
- **Prefect**: https://github.com/PrefectHQ/prefect
- **Mage**: https://github.com/mage-ai/mage-ai
- **Apache Airflow**: https://github.com/apache/airflow
- **Dagster**: https://github.com/dagster-io/dagster

### Articulos y comparaciones
- [MLOps Zoomcamp - Module 3: Orchestration](https://github.com/DataTalksClub/mlops-zoomcamp/tree/main/03-orchestration)
- [Prefect vs Airflow: What's the Difference?](https://docs.prefect.io/latest/resources/airflow-vs-prefect/)
- [Mage vs Airflow](https://docs.mage.ai/about/comparison/airflow)

### Conclusion

> **La herramienta no importa tanto como entender el concepto.**
> La orquestacion se trata de: automatizar, observar, y hacer resilientes
> tus pipelines de datos y ML. Ya sea con Prefect, Mage, Airflow o cualquier
> otra herramienta, los principios son los mismos:
>
> 1. **Definir pasos claros** (tasks/blocks)
> 2. **Conectarlos en un flujo** (flow/pipeline)
> 3. **Automatizar la ejecucion** (scheduling/triggers)
> 4. **Observar y reaccionar** (dashboard/logs/artefactos)
> 5. **Manejar errores** (retries/alertas)